In [ ]:
# V2 — Aerodynamic Drag
# Tennis Serve Admissibility Project

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# -------------------------
# Physical constants
# -------------------------

G = 9.81                  # gravitational acceleration (m/s^2)

# Tennis ball properties
BALL_MASS = 0.0575        # kg
BALL_DIAMETER = 0.067     # m
AIR_DENSITY = 1.21        # kg/m^3
DRAG_COEFFICIENT = 0.55

# Cross-sectional area
BALL_AREA = np.pi * (BALL_DIAMETER / 2)**2

# -------------------------
# Court geometry
# -------------------------

NET_DISTANCE = 11.885          # m
SERVICE_LINE_DISTANCE = 18.285 # m
NET_HEIGHT_CENTER = 0.914      # m

# -------------------------
# Initial serve conditions
# -------------------------

CONTACT_HEIGHT = 3.0           # m
SERVE_SPEED_KMH = 200.0
SERVE_SPEED_MS = SERVE_SPEED_KMH / 3.6

print("V2 environment ready.")
print("-" * 45)
print(f"Ball mass:            {BALL_MASS:.4f} kg")
print(f"Ball diameter:        {BALL_DIAMETER:.4f} m")
print(f"Air density:          {AIR_DENSITY:.2f} kg/m³")
print(f"Drag coefficient:     {DRAG_COEFFICIENT:.2f}")
print(f"Ball cross-section:   {BALL_AREA:.6f} m²")
print(f"Serve speed:          {SERVE_SPEED_MS:.2f} m/s")

In [ ]:
# V2 Cell 2 — Aerodynamic Drag Force

def drag_force(vx, vz):
    """
    Calculate the 2D aerodynamic drag force.

    Drag always acts opposite to the direction of motion.

    Parameters
    ----------
    vx : float
        Horizontal velocity (m/s)
    vz : float
        Vertical velocity (m/s)

    Returns
    -------
    Fx, Fz : float
        Drag force components (N)
    """

    speed = np.sqrt(vx**2 + vz**2)

    Fx = -0.5 * AIR_DENSITY * DRAG_COEFFICIENT * BALL_AREA * speed * vx
    Fz = -0.5 * AIR_DENSITY * DRAG_COEFFICIENT * BALL_AREA * speed * vz

    return Fx, Fz


# Test at the initial serve velocity
vx0 = SERVE_SPEED_MS * np.cos(np.radians(6))
vz0 = SERVE_SPEED_MS * np.sin(np.radians(6))

Fx_drag, Fz_drag = drag_force(vx0, vz0)

drag_magnitude = np.sqrt(Fx_drag**2 + Fz_drag**2)

print("Drag-force sanity check")
print("-" * 45)
print(f"Initial speed:       {SERVE_SPEED_MS:.3f} m/s")
print(f"Horizontal velocity: {vx0:.3f} m/s")
print(f"Vertical velocity:   {vz0:.3f} m/s")
print()
print(f"Drag force Fx:       {Fx_drag:.4f} N")
print(f"Drag force Fz:       {Fz_drag:.4f} N")
print(f"Drag magnitude:      {drag_magnitude:.4f} N")

In [ ]:
# V2 Cell 3 — Equations of Motion With Aerodynamic Drag

def trajectory_with_drag(t, state):
    """
    2D tennis-ball trajectory with gravity and aerodynamic drag.

    State vector:
        [x, z, vx, vz]

    Equations:
        dx/dt  = vx
        dz/dt  = vz
        dvx/dt = Fx_drag / m
        dvz/dt = -g + Fz_drag / m
    """

    x, z, vx, vz = state

    # Current speed
    speed = np.sqrt(vx**2 + vz**2)

    # Aerodynamic drag force
    Fx_drag = (
        -0.5
        * AIR_DENSITY
        * DRAG_COEFFICIENT
        * BALL_AREA
        * speed
        * vx
    )

    Fz_drag = (
        -0.5
        * AIR_DENSITY
        * DRAG_COEFFICIENT
        * BALL_AREA
        * speed
        * vz
    )

    # Accelerations
    ax = Fx_drag / BALL_MASS
    az = -G + Fz_drag / BALL_MASS

    return np.array([
        vx,
        vz,
        ax,
        az
    ])


# Check the initial acceleration
initial_state = np.array([
    0.0,
    CONTACT_HEIGHT,
    vx0,
    vz0
])

initial_derivative = trajectory_with_drag(
    0.0,
    initial_state
)

print("Initial acceleration with drag")
print("-" * 45)
print(f"Horizontal acceleration: {initial_derivative[2]:.3f} m/s²")
print(f"Vertical acceleration:   {initial_derivative[3]:.3f} m/s²")

In [ ]:
# V2 Cell 4 — Production Trajectory Solver

def landing_event(t, state):
    """
    Event triggered when the ball reaches court level (z = 0).
    """
    return state[1]


# Stop the integration when the ball is moving downward
landing_event.terminal = True
landing_event.direction = -1


# Initial launch angle
theta_drag = np.radians(6.0)

vx0_drag = SERVE_SPEED_MS * np.cos(theta_drag)
vz0_drag = SERVE_SPEED_MS * np.sin(theta_drag)

initial_state_drag = np.array([
    0.0,                 # x
    CONTACT_HEIGHT,      # z
    vx0_drag,            # vx
    vz0_drag             # vz
])


# Integrate the trajectory
solution_drag = solve_ivp(
    trajectory_with_drag,
    t_span=(0.0, 3.0),
    y0=initial_state_drag,
    events=landing_event,
    rtol=1e-9,
    atol=1e-11,
    dense_output=True,
    max_step=0.001
)

# Extract trajectory
t_drag = solution_drag.t
x_drag = solution_drag.y[0]
z_drag = solution_drag.y[1]
vx_drag = solution_drag.y[2]
vz_drag = solution_drag.y[3]


# Landing information
if len(solution_drag.t_events[0]) > 0:

    landing_time = solution_drag.t_events[0][0]
    landing_state = solution_drag.y_events[0][0]

    landing_x = landing_state[0]
    landing_z = landing_state[1]

    print("Drag trajectory")
    print("-" * 45)
    print(f"Landing time:        {landing_time:.6f} s")
    print(f"Landing x-position:  {landing_x:.6f} m")
    print(f"Landing height:      {landing_z:.6e} m")
    print(f"Final horizontal v:  {landing_state[2]:.6f} m/s")
    print(f"Final vertical v:    {landing_state[3]:.6f} m/s")

else:
    print("The ball did not reach the court within the integration interval.")

In [ ]:
# V2 Cell 5 — Compare Drag vs. No-Drag Trajectories

# Time points from the drag solution
t_drag = solution_drag.t
x_drag = solution_drag.y[0]
z_drag = solution_drag.y[1]

# No-drag analytical trajectory
theta_drag = np.radians(6)

vx0_test = SERVE_SPEED_MS * np.cos(theta_drag)
vz0_test = SERVE_SPEED_MS * np.sin(theta_drag)

x_no_drag = vx0_test * t_drag
z_no_drag = CONTACT_HEIGHT + vz0_test * t_drag - 0.5 * G * t_drag**2

# Plot comparison
plt.figure(figsize=(10, 5))

plt.plot(
    x_no_drag,
    z_no_drag,
    label="No drag"
)

plt.plot(
    x_drag,
    z_drag,
    label="With drag"
)

plt.axhline(
    0,
    linestyle="--",
    label="Court surface"
)

plt.xlabel("Horizontal position x (m)")
plt.ylabel("Height z (m)")
plt.title("Effect of Aerodynamic Drag on Tennis-Ball Trajectory")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# V2 Cell 6 — Numerical Accuracy / Tolerance Study

tolerances = [1e-6, 1e-8, 1e-10, 1e-12]

results = []

for tol in tolerances:

    sol = solve_ivp(
        trajectory_with_drag,
        t_span=(0.0, 3.0),
        y0=initial_state_drag,
        events=landing_event,
        rtol=tol,
        atol=tol * 1e-2,
        dense_output=True,
        max_step=0.001
    )

    if len(sol.t_events[0]) == 0:
        raise RuntimeError(
            f"Landing event was not detected for tolerance {tol:.0e}."
        )

    landing_time = sol.t_events[0][0]
    landing_state = sol.y_events[0][0]

    results.append([
        tol,
        landing_time,
        landing_state[0],
        landing_state[2],
        landing_state[3]
    ])

results = np.array(results)

print("V2 SOLVER TOLERANCE STUDY")
print("---------------------------------------------")
print("Tolerance      Time (s)       X landing (m)")

for row in results:
    print(
        f"{row[0]:.0e}       "
        f"{row[1]:.9f}      "
        f"{row[2]:.9f}"
    )

# Difference from the tightest solution
reference_x = results[-1, 2]

print("\nDifference from tightest solution")
print("---------------------------------------------")

for row in results:
    difference = abs(row[2] - reference_x)

    print(
        f"Tolerance {row[0]:.0e}: "
        f"{difference:.3e} m"
    )

In [ ]:
# V2 Cell 7 — Quantitative Drag Comparison

# Drag case
drag_landing_time = solution_drag.t_events[0][0]
drag_landing_x = solution_drag.y_events[0][0][0]
drag_final_speed = np.sqrt(
    solution_drag.y_events[0][0][2]**2 +
    solution_drag.y_events[0][0][3]**2
)

# No-drag landing time
# Solve z(t) = 0 analytically:
# z0 + vz0*t - 0.5*G*t^2 = 0

a = -0.5 * G
b = vz0
c = CONTACT_HEIGHT

roots = np.roots([a, b, c])
no_drag_landing_time = max(roots)

no_drag_landing_x = vx0_drag * no_drag_landing_time
no_drag_final_speed = np.sqrt(
    vx0_drag**2 +
    (vz0_drag - G * no_drag_landing_time)**2
)

# Percentage differences
range_reduction = (
    (no_drag_landing_x - drag_landing_x)
    / no_drag_landing_x
    * 100
)

speed_reduction = (
    (SERVE_SPEED_MS - drag_final_speed)
    / SERVE_SPEED_MS
    * 100
)

print("V2 DRAG QUANTIFICATION")
print("---------------------------------------------")

print(f"No-drag landing time:     {no_drag_landing_time:.6f} s")
print(f"Drag landing time:        {drag_landing_time:.6f} s")

print(f"\nNo-drag landing x:        {no_drag_landing_x:.6f} m")
print(f"Drag landing x:           {drag_landing_x:.6f} m")

print(f"\nRange reduction:           {range_reduction:.2f}%")

print(f"\nInitial speed:             {SERVE_SPEED_MS:.6f} m/s")
print(f"Final speed with drag:    {drag_final_speed:.6f} m/s")

print(f"\nSpeed reduction:           {speed_reduction:.2f}%")

In [ ]:
# V2 Cell 8 — Drag Coefficient Sensitivity

cd_values = [0.45, 0.50, 0.55, 0.60, 0.65]

cd_results = []

for cd in cd_values:

    def trajectory_with_test_cd(t, state):
        x, z, vx, vz = state

        speed = np.sqrt(vx**2 + vz**2)

        Fx_drag = (
            -0.5
            * AIR_DENSITY
            * cd
            * BALL_AREA
            * speed
            * vx
        )

        Fz_drag = (
            -0.5
            * AIR_DENSITY
            * cd
            * BALL_AREA
            * speed
            * vz
        )

        ax = Fx_drag / BALL_MASS
        az = -G + Fz_drag / BALL_MASS

        return np.array([vx, vz, ax, az])

    sol = solve_ivp(
        trajectory_with_test_cd,
        t_span=(0.0, 3.0),
        y0=initial_state,
        events=landing_event,
        rtol=1e-10,
        atol=1e-12,
        max_step=0.001
    )

    if len(sol.t_events[0]) == 0:
        raise RuntimeError(
            f"Landing event was not detected for Cd = {cd:.2f}."
        )

    landing_time = sol.t_events[0][0]
    landing_state = sol.y_events[0][0]

    cd_results.append([
        cd,
        landing_time,
        landing_state[0]
    ])

print("V2 DRAG COEFFICIENT SENSITIVITY")
print("---------------------------------------------")
print("Cd          Landing time (s)    Landing x (m)")

for cd, time, x in cd_results:
    print(f"{cd:.2f}        {time:.9f}         {x:.6f}")

In [ ]:
# V2 Cell 9 — Verification Summary

# Check monotonic behavior
landing_distances = np.array([row[2] for row in cd_results])

monotonic_drag_response = np.all(np.diff(landing_distances) < 0)

# Check numerical stability
tolerance_stable = np.allclose(
    results[:, 2],
    results[-1, 2],
    atol=1e-6
)

print("=" * 55)
print("V2 AERODYNAMIC DRAG VERIFICATION")
print("=" * 55)

print(f"Baseline Cd:                 {DRAG_COEFFICIENT:.2f}")
print(f"Baseline landing distance:   {drag_landing_x:.6f} m")
print(f"Baseline landing time:       {drag_landing_time:.6f} s")

print("\nNumerical stability:")
print("---------------------------------------------")
print(f"Tolerance study stable:      {'PASS' if tolerance_stable else 'FAIL'}")

print("\nPhysical sensitivity:")
print("---------------------------------------------")
print(
    f"Landing distance decreases with Cd: "
    f"{'PASS' if monotonic_drag_response else 'FAIL'}"
)

print("\nModel limitation:")
print("---------------------------------------------")
print("Cd = 0.55 is treated as a provisional parameter.")
print("It has not yet been independently calibrated.")

if tolerance_stable and monotonic_drag_response:
    print("\nV2 VERIFICATION: PASS")
else:
    print("\nV2 VERIFICATION: REVIEW REQUIRED")